# Machine Learning — Practical Assignment 1

Complete solutions to all **7 assignments** in the supplied PDF. Run each cell from top to bottom. Data are fetched from **UCI Machine Learning Repository**, so the first execution needs internet access. The notebook uses NumPy, pandas, Matplotlib and scikit-learn; code from scratch is used wherever requested.

**Evaluation protocol:** For assignments 3–7, 60% train / 20% validation / 20% test (`random_state=42`), stratified for classification. Fit preprocessing only on training data; choose models and thresholds on validation; evaluate test data after selection.

In [ ]:
# Run this once if necessary (uncomment in Jupyter / Colab):
# %pip install ucimlrepo numpy pandas matplotlib scikit-learn scipy

In [ ]:
import sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, scipy, ucimlrepo
from scipy.stats import norm, beta
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_auc_score,
                             roc_curve)
from ucimlrepo import fetch_ucirepo
SEED = 42
rng = np.random.default_rng(SEED)
print('Python:', sys.version.split()[0], 'NumPy:', np.__version__, 'pandas:', pd.__version__,
      'Matplotlib:', plt.matplotlib.__version__ if hasattr(plt, 'matplotlib') else 'installed',
      'scikit-learn:', sklearn.__version__, 'SciPy:', scipy.__version__,
      'ucimlrepo:', getattr(ucimlrepo, '__version__', 'installed'))
print('Seed:', SEED)

## Load the three UCI datasets

UCI IDs: Iris **53**, Auto MPG **9**, Breast Cancer Wisconsin (Diagnostic) **17**. Keep identifier columns out of predictors.

In [ ]:
iris_repo = fetch_ucirepo(id=53)
auto_repo = fetch_ucirepo(id=9)
cancer_repo = fetch_ucirepo(id=17)
X_iris = iris_repo.data.features.copy()
y_iris = iris_repo.data.targets.iloc[:, 0].copy()
X_auto_all = auto_repo.data.features.copy()
y_auto = pd.to_numeric(auto_repo.data.targets.iloc[:, 0], errors='coerce')
X_cancer_all = cancer_repo.data.features.copy()
y_cancer_raw = cancer_repo.data.targets.iloc[:, 0].copy()

def clean_columns(frame):
    frame = frame.copy()
    frame.columns = (frame.columns.astype(str).str.strip().str.lower()
                     .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
    return frame

X_iris = clean_columns(X_iris)
X_auto_all = clean_columns(X_auto_all)
X_cancer_all = clean_columns(X_cancer_all)
# Remove any identifiers if present; choose only numerical predictors for cancer/iris.
X_iris = X_iris.select_dtypes(include='number')
X_cancer = X_cancer_all.drop(columns=['id', 'sample_code_number'], errors='ignore').select_dtypes(include='number')
# Normalize UCI label strings; 1 = malignant, 0 = benign.
labels = y_cancer_raw.astype(str).str.strip().str.lower()
label_map = {'m': 1, 'malignant': 1, 'b': 0, 'benign': 0}
y_cancer = labels.map(label_map)
assert y_cancer.notna().all(), f'Unknown label(s): {labels.unique()}'
y_cancer = y_cancer.astype(int)

# Accept small differences in the UCI column spellings.
requested = ['weight', 'horsepower', 'displacement', 'acceleration', 'model_year']
if 'model_year' not in X_auto_all.columns and 'model_yr' in X_auto_all.columns:
    X_auto_all = X_auto_all.rename(columns={'model_yr': 'model_year'})
assert set(requested).issubset(X_auto_all.columns), f'Auto MPG columns: {list(X_auto_all.columns)}'
X_auto = X_auto_all[requested].apply(pd.to_numeric, errors='coerce')
valid = y_auto.notna()
X_auto, y_auto = X_auto.loc[valid].reset_index(drop=True), y_auto.loc[valid].reset_index(drop=True)
print('Iris:', X_iris.shape, 'Auto MPG:', X_auto.shape, 'Cancer:', X_cancer.shape)
print('Iris species:', list(y_iris.unique()))
print('Auto missing values:\n', X_auto.isna().sum())
print('Cancer labels (1=malignant):\n', y_cancer.value_counts().sort_index())

## 1. Maximum Likelihood Estimation — Iris

For observations $x_1,\ldots,x_n$ with normal distribution $N(\mu,\sigma^2)$:

$$L(\mu,\sigma^2)=\prod_{i=1}^{n}\frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{(x_i-\mu)^2}{2\sigma^2}\right)$$

$$\ell(\mu,\sigma^2)=-\frac n2\log(2\pi\sigma^2)-\frac{\sum_i(x_i-\mu)^2}{2\sigma^2}$$

MLEs: $\hat\mu=\overline{x}$, $\hat\sigma^2=\sum_i(x_i-\bar{x})^2/n$. The unbiased variance uses $n-1$.

In [ ]:
sepal_col = next(c for c in X_iris.columns if 'sepal' in c and 'length' in c)
sepal = X_iris[sepal_col].to_numpy(dtype=float)
species = y_iris.astype(str).to_numpy()

def gaussian_likelihood(x, mu, variance):
    return np.prod(norm.pdf(x, loc=mu, scale=np.sqrt(variance)))

def gaussian_log_likelihood(x, mu, variance):
    x = np.asarray(x, dtype=float)
    return -len(x)/2 * np.log(2*np.pi*variance) - np.sum((x-mu)**2)/(2*variance)

records = []
fig, axs = plt.subplots(1, len(np.unique(species)), figsize=(15, 4))
for ax, sp in zip(axs, np.unique(species)):
    x = sepal[species == sp]
    mu, var_mle = np.mean(x), np.var(x, ddof=0)
    var_unbiased = np.var(x, ddof=1)
    records.append({'species': sp, 'n': len(x), 'mean_MLE': mu,
                    'variance_MLE': var_mle, 'variance_unbiased': var_unbiased,
                    'log_likelihood_at_MLE': gaussian_log_likelihood(x, mu, var_mle)})
    grid = np.linspace(x.min()-0.5, x.max()+0.5, 250)
    ax.hist(x, bins=10, density=True, alpha=0.55, edgecolor='black')
    ax.plot(grid, norm.pdf(grid, mu, np.sqrt(var_mle)), lw=2)
    ax.set(title=str(sp), xlabel='Sepal length (cm)', ylabel='Density')
plt.suptitle('Species histograms with fitted normal densities')
plt.tight_layout(); plt.show()
param_table = pd.DataFrame(records)
display(param_table.round(4))
print('MLE variance divides by n, so is slightly smaller than unbiased variance (n−1).')
print('Inspect histogram shape against the smooth Gaussian curve to assess the approximation.')

In [ ]:
sizes = [5, 10, 20, 40]
subset_records = []
for sp in np.unique(species):
    values = sepal[species == sp]
    for size in sizes:
        for rep in range(30):
            subset = rng.choice(values, size=size, replace=False)
            subset_records.append({'species': sp, 'size': size, 'repeat': rep,
                                   'estimated_mean': np.mean(subset),
                                   'estimated_variance': np.var(subset, ddof=0)})
subsets = pd.DataFrame(subset_records)
summary1 = subsets.groupby(['species', 'size'])[['estimated_mean', 'estimated_variance']].agg(['mean', 'std'])
display(summary1.round(4))
fig, axs = plt.subplots(1, 2, figsize=(12,4))
for sp, sub in subsets.groupby('species'):
    for ax, col, title in [(axs[0], 'estimated_mean', 'Mean estimate variability'),
                           (axs[1], 'estimated_variance', 'Variance estimate variability')]:
        s = sub.groupby('size')[col].std()
        ax.plot(s.index, s.values, marker='o', label=sp)
        ax.set(xlabel='Subset size', ylabel='Std. dev. across 30 repeats', title=title)
for ax in axs: ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print('Larger subsets generally produce more stable estimates (lower across-repeat SD).')

## 2. Maximum A Posteriori Estimation — Breast Cancer

For $k$ malignant cases in $n$ observations, with prior $p\sim\mathrm{Beta}(a,b)$:

- Likelihood: $p^k(1-p)^{n-k}$; MLE $k/n$.
- Posterior: $p\mid y\sim\mathrm{Beta}(a+k,b+n-k)$.
- MAP (when both posterior parameters exceed 1): $(a+k-1)/(a+b+n-2)$.
- Posterior mean: $(a+k)/(a+b+n)$ (not the MAP).

In [ ]:
priors = [(2,2), (2,8), (8,2)]
labels_bin = y_cancer.to_numpy()
reference = labels_bin.mean()

def bernoulli_mle(y): return np.mean(y)
def beta_posterior(y, a, b):
    k = int(np.sum(y)); return a+k, b+len(y)-k

def bernoulli_map(y, a, b):
    A, B = beta_posterior(y, a, b)
    if A > 1 and B > 1: return (A-1)/(A+B-2)
    if A <= 1 and B > 1: return 0.0
    if B <= 1 and A > 1: return 1.0
    return np.nan  # boundary-mode ambiguity, not used with chosen priors

chosen = rng.choice(labels_bin, size=30, replace=False)
grid = np.linspace(0.001, 0.999, 400)
plt.figure(figsize=(9,5))
for a,b in priors:
    A,B = beta_posterior(chosen,a,b)
    plt.plot(grid, beta.pdf(grid,a,b), '--', label=f'Prior Beta({a},{b})')
    plt.plot(grid, beta.pdf(grid,A,B), label=f'Posterior Beta({A},{B})')
plt.axvline(chosen.mean(), color='black', alpha=.7, label='Subset MLE')
plt.title('Prior and posterior densities for one 30-item subset'); plt.xlabel('Malignant probability p')
plt.ylabel('Density'); plt.legend(); plt.show()

map_rows = []
for size in [10,30,100,300]:
    for rep in range(30):
        selection = rng.choice(labels_bin, size=size, replace=False)
        map_rows.append({'n': size, 'repeat': rep, 'method': 'MLE', 'estimate': bernoulli_mle(selection)})
        for a,b in priors:
            A,B = beta_posterior(selection,a,b)
            map_rows.append({'n': size, 'repeat': rep, 'method': f'MAP Beta({a},{b})',
                             'estimate': bernoulli_map(selection,a,b)})
map_df = pd.DataFrame(map_rows)
display(map_df.groupby(['n','method']).estimate.agg(['mean','std']).round(4))
plt.figure(figsize=(10,5))
for method, sub in map_df.groupby('method'):
    g = sub.groupby('n').estimate.agg(['mean','std'])
    plt.errorbar(g.index, g['mean'], yerr=g['std'], marker='o', capsize=3, label=method)
plt.axhline(reference, color='black', linestyle=':', label='Full dataset proportion (descriptive)')
plt.xlabel('Subset size'); plt.ylabel('Estimated malignant proportion'); plt.title('MLE vs MAP (mean ± SD over 30 repeats)')
plt.legend(); plt.grid(alpha=.3); plt.show()
print('Full-data malignant proportion (descriptive, not population truth):', round(reference,4))
print('Prior influence is strongest for small n; observed counts dominate as n increases.')
print('For example, posterior mean and MAP for chosen subset under Beta(2,8):',
      (sum(chosen)+2)/(len(chosen)+10), bernoulli_map(chosen,2,8))

## Shared train / validation / test splits

First isolate 20% test; then split the other 80% into 60% train and 20% validation. Stratification is required for classification tasks.

In [ ]:
def split_60_20_20(X, y, classification=False):
    strat = y if classification else None
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=strat)
    strat_temp = y_temp if classification else None
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=strat_temp)
    return X_train, X_val, X_test, y_train, y_val, y_test

auto_tr, auto_va, auto_te, mpg_tr, mpg_va, mpg_te = split_60_20_20(X_auto, y_auto)
cancer_tr, cancer_va, cancer_te, cy_tr, cy_va, cy_te = split_60_20_20(X_cancer, y_cancer, True)
iris_tr, iris_va, iris_te, iy_tr, iy_va, iy_te = split_60_20_20(X_iris, y_iris, True)
print('Auto train/val/test:', len(auto_tr), len(auto_va), len(auto_te))
print('Cancer train/val/test:', len(cancer_tr), len(cancer_va), len(cancer_te))
print('Iris train/val/test:', len(iris_tr), len(iris_va), len(iris_te))

## 3. Linear Regression — Auto MPG

Simple model: weight only. Multiple model: weight, horsepower, displacement, acceleration, model year. Fit median imputation on train **only**; use the normal equation / pseudoinverse for coefficients from scratch. Compare to scikit-learn on the held-out sets.

In [ ]:
print('Missing values (full Auto MPG dataset):')
display(X_auto.isna().sum().to_frame('missing'))
fig, axs = plt.subplots(1, 2, figsize=(11,4))
axs[0].scatter(X_auto.weight, y_auto, s=12, alpha=.55)
axs[0].set(xlabel='Weight', ylabel='MPG', title='Weight vs MPG')
axs[1].scatter(X_auto.horsepower, y_auto, s=12, alpha=.55)
axs[1].set(xlabel='Horsepower', ylabel='MPG', title='Horsepower vs MPG')
plt.tight_layout(); plt.show()

imputer_auto = SimpleImputer(strategy='median').fit(auto_tr)
Xa_tr = imputer_auto.transform(auto_tr); Xa_va = imputer_auto.transform(auto_va); Xa_te = imputer_auto.transform(auto_te)
ytr, yva, yte = mpg_tr.to_numpy(), mpg_va.to_numpy(), mpg_te.to_numpy()

def fit_ols_from_scratch(X, y):
    Xdesign = np.column_stack([np.ones(len(X)), X])
    return np.linalg.pinv(Xdesign) @ np.asarray(y)

def predict_ols(X, coef):
    return np.column_stack([np.ones(len(X)), X]) @ coef

def regression_metrics(y_true, pred):
    return {'MAE': mean_absolute_error(y_true, pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, pred)), 'R2': r2_score(y_true, pred)}

weight_idx = requested.index('weight')
simple_coef = fit_ols_from_scratch(Xa_tr[:,[weight_idx]], ytr)
multi_coef = fit_ols_from_scratch(Xa_tr, ytr)
sk_simple = LinearRegression().fit(Xa_tr[:,[weight_idx]], ytr)
sk_multi = LinearRegression().fit(Xa_tr, ytr)
print('Simple scratch: intercept, weight slope =', simple_coef)
print('Multiple scratch:')
display(pd.Series(multi_coef, index=['intercept']+requested, name='coefficient'))
comparison3 = []
models3 = {
    'Simple scratch': (predict_ols(Xa_va[:,[weight_idx]], simple_coef), predict_ols(Xa_te[:,[weight_idx]], simple_coef)),
    'Simple sklearn': (sk_simple.predict(Xa_va[:,[weight_idx]]), sk_simple.predict(Xa_te[:,[weight_idx]])),
    'Multiple scratch': (predict_ols(Xa_va, multi_coef), predict_ols(Xa_te, multi_coef)),
    'Multiple sklearn': (sk_multi.predict(Xa_va), sk_multi.predict(Xa_te))
}
for name,(predval,predtest) in models3.items():
    comparison3.append({'model':name, **{'val_'+k:v for k,v in regression_metrics(yva,predval).items()},
                       **{'test_'+k:v for k,v in regression_metrics(yte,predtest).items()}})
display(pd.DataFrame(comparison3).round(4))

fig, axs = plt.subplots(1,3,figsize=(16,4))
axs[0].scatter(Xa_tr[:,weight_idx], ytr, s=12, alpha=.5)
wgrid = np.linspace(Xa_tr[:,weight_idx].min(), Xa_tr[:,weight_idx].max(),200)
axs[0].plot(wgrid, simple_coef[0]+simple_coef[1]*wgrid, 'r')
axs[0].set(title='Simple regression fitted line', xlabel='Weight', ylabel='MPG')
pred = predict_ols(Xa_te,multi_coef)
axs[1].scatter(yte,pred,alpha=.65); lim=[min(yte.min(),pred.min()),max(yte.max(),pred.max())]
axs[1].plot(lim,lim,'k--'); axs[1].set(title='Actual vs predicted (multiple)',xlabel='Actual MPG',ylabel='Predicted MPG')
axs[2].scatter(pred,yte-pred,alpha=.65); axs[2].axhline(0,color='black',linestyle='--')
axs[2].set(title='Residual vs predicted (multiple)',xlabel='Predicted MPG',ylabel='Residual (actual − predicted)')
plt.tight_layout();plt.show()
print('Weight slope in simple model:', round(simple_coef[1],6),
      'MPG per unit weight. Negative means heavier cars tend to have lower predicted MPG.')
print('Compare validation and test RMSE in the table to judge improvement from additional predictors.')

## 4. Logistic Regression — Breast Cancer

Train an **unregularized** binary logistic regression model from scratch with intercept, stable `logaddexp` binary cross-entropy and batch gradient descent. Tune learning rate and epochs on validation. The positive class is **malignant = 1**.

In [ ]:
cancer_imp = SimpleImputer(strategy='median').fit(cancer_tr)
c_tr0 = cancer_imp.transform(cancer_tr)
c_va0 = cancer_imp.transform(cancer_va)
c_te0 = cancer_imp.transform(cancer_te)
c_scaler = StandardScaler().fit(c_tr0)
C_tr, C_va, C_te = (c_scaler.transform(a) for a in [c_tr0,c_va0,c_te0])
Y_tr,Y_va,Y_te = (s.to_numpy().astype(int) for s in [cy_tr,cy_va,cy_te])

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1/(1+np.exp(-z[pos]))
    ez = np.exp(z[~pos]); out[~pos] = ez/(1+ez)
    return out

def logistic_loss(X, y, w, bias):
    z = X@w + bias
    return np.mean(np.logaddexp(0,z)-y*z)

def train_logistic(X, y, lr, epochs):
    w = np.zeros(X.shape[1]); b = 0.; history=[]
    for epoch in range(epochs):
        p = sigmoid(X@w+b)
        error = p-y
        w -= lr * (X.T@error)/len(y)
        b -= lr * error.mean()
        history.append(logistic_loss(X,y,w,b))
    return w,b,history

log_trials=[]
for lr in [0.001,0.01,0.1,0.5]:
    for epochs in [200,500,1000,2000]:
        w,b,hist = train_logistic(C_tr,Y_tr,lr,epochs)
        pv=sigmoid(C_va@w+b)
        log_trials.append({'lr':lr,'epochs':epochs,'val_log_loss':logistic_loss(C_va,Y_va,w,b),
                           'val_f1_at_0_5':f1_score(Y_va,(pv>=0.5).astype(int)),
                           'w':w,'b':b,'history':hist})
# Primary criterion: lowest validation log loss; deterministic tie-break: fewer epochs/lower lr.
selected4=min(log_trials,key=lambda r:(r['val_log_loss'],r['epochs'],r['lr']))
print('Best from-scratch parameters:', {k:selected4[k] for k in ['lr','epochs','val_log_loss','val_f1_at_0_5']})
display(pd.DataFrame([{k:r[k] for k in ['lr','epochs','val_log_loss','val_f1_at_0_5']} for r in log_trials]).round(4))
plt.plot(selected4['history']); plt.title('Selected logistic model training loss');plt.xlabel('Epoch');plt.ylabel('Binary cross-entropy');plt.show()

# Scikit-learn comparison: no regularization, matching training-only standardized predictors.
sk_log = LogisticRegression(penalty=None, max_iter=10000, random_state=SEED).fit(C_tr,Y_tr)
p_val_scratch=sigmoid(C_va@selected4['w']+selected4['b'])
p_val_sklearn=sk_log.predict_proba(C_va)[:,1]
print('scikit-learn regularization: penalty=None (unregularized)')
display(pd.DataFrame([{'model':'Scratch','validation ROC-AUC':roc_auc_score(Y_va,p_val_scratch),
                       'validation F1 @ 0.5':f1_score(Y_va,p_val_scratch>=0.5)},
                      {'model':'sklearn','validation ROC-AUC':roc_auc_score(Y_va,p_val_sklearn),
                       'validation F1 @ 0.5':f1_score(Y_va,p_val_sklearn>=0.5)}]).round(4))

threshold_rows=[]
for t in [0.3,0.5,0.7]:
    yp=(p_val_scratch>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(Y_va,yp,labels=[0,1]).ravel()
    threshold_rows.append({'threshold':t,'accuracy':accuracy_score(Y_va,yp),
        'precision':precision_score(Y_va,yp,zero_division=0),
        'recall':recall_score(Y_va,yp,zero_division=0),
        'F1':f1_score(Y_va,yp,zero_division=0),'FP':fp,'FN':fn})
threshold_df=pd.DataFrame(threshold_rows)
display(threshold_df.round(4))
# Criterion: maximum validation F1; ties favor lower threshold (higher malignant recall).
threshold4=float(threshold_df.sort_values(['F1','threshold'],ascending=[False,True]).iloc[0]['threshold'])
p_test=sigmoid(C_te@selected4['w']+selected4['b'])
yp_test=(p_test>=threshold4).astype(int)
print('Selected threshold using validation F1:',threshold4)
print('Test metrics (scratch):', {'accuracy':accuracy_score(Y_te,yp_test),
    'precision':precision_score(Y_te,yp_test,zero_division=0),
    'recall':recall_score(Y_te,yp_test,zero_division=0),
    'F1':f1_score(Y_te,yp_test,zero_division=0),
    'ROC-AUC':roc_auc_score(Y_te,p_test)})
fig,axs=plt.subplots(1,2,figsize=(11,4))
fpr,tpr,_=roc_curve(Y_te,p_test)
axs[0].plot(fpr,tpr,label=f'AUC = {roc_auc_score(Y_te,p_test):.3f}')
axs[0].plot([0,1],[0,1],'k--');axs[0].set(title='Test ROC',xlabel='False-positive rate',ylabel='True-positive rate');axs[0].legend()
ConfusionMatrixDisplay.from_predictions(Y_te,yp_test,display_labels=['Benign','Malignant'],ax=axs[1],colorbar=False)
axs[1].set_title('Test confusion matrix')
plt.tight_layout();plt.show()
print('Lower threshold usually raises recall (fewer FN) but increases FP; higher threshold usually does the reverse.')

## 5. Gradient Descent and its Variants — Auto MPG

Use the same **five predictors** as assignment 3, training-only imputation + scaling, and objective $J(w,b)=\frac1{2m}\sum_i (wx_i+b-y_i)^2$. Compare batch, SGD, and mini-batch (16, 32) with learning rates 0.0001, 0.001 and 0.01, up to 500 epochs. Record training objective **after each epoch**, elapsed time, and number of updates. Randomly shuffle sample order each epoch for SGD / mini-batch.

In [ ]:
# Reuse the training-only median-imputed matrices from assignment 3.
scaler_auto = StandardScaler().fit(Xa_tr)
G_tr,G_va,G_te = (scaler_auto.transform(a) for a in [Xa_tr,Xa_va,Xa_te])

def train_gradient_descent(X, y, batch_size, lr, epochs=500, seed=SEED):
    generator=np.random.default_rng(seed)
    m,p=X.shape; w=np.zeros(p); b=0.0
    losses=[]; elapsed=[]; updates=0
    start=time.perf_counter()
    for epoch in range(epochs):
        order=np.arange(m) if batch_size==m else generator.permutation(m)
        for start_idx in range(0,m,batch_size):
            ids=order[start_idx:start_idx+batch_size]
            error=(X[ids]@w+b)-y[ids]
            w-=lr*(X[ids].T@error)/len(ids)
            b-=lr*error.mean()
            updates+=1
        err=X@w+b-y
        loss=float(np.mean(err**2)/2)
        losses.append(loss)
        elapsed.append(time.perf_counter()-start)
        if not np.isfinite(loss) or loss>1e12:
            return w,b,losses,elapsed,updates,True
    return w,b,losses,elapsed,updates,False

runs5=[]
for label,bs in [('Batch',len(G_tr)),('SGD',1),('Mini-16',16),('Mini-32',32)]:
    for lr in [0.0001,0.001,0.01]:
        w,b,losses,elapsed,updates,divergent=train_gradient_descent(G_tr,ytr,bs,lr)
        val_rmse=(np.sqrt(np.mean((G_va@w+b-yva)**2)) if not divergent else np.inf)
        runs5.append({'method':label,'batch_size':bs,'lr':lr,'w':w,'b':b,
                      'losses':losses,'elapsed':elapsed,'updates':updates,
                      'divergent':divergent,'val_RMSE':val_rmse})

# Validation selection per method. Compare final test RMSE only AFTER choosing configurations.
chosen5=[]
for method in ['Batch','SGD','Mini-16','Mini-32']:
    candidates=[r for r in runs5 if r['method']==method and not r['divergent']]
    best=min(candidates,key=lambda r:(r['val_RMSE'],r['lr']))
    chosen5.append(best)

result5=[]
for r in runs5:
    test_rmse=np.nan
    if any(r is chosen for chosen in chosen5):
        test_rmse=float(np.sqrt(np.mean((G_te@r['w']+r['b']-yte)**2)))
    result5.append({'method':r['method'],'learning_rate':r['lr'],
                    'batch_size':r['batch_size'],'updates':r['updates'],
                    'runtime_seconds':r['elapsed'][-1],
                    'final_train_loss':r['losses'][-1],
                    'validation_RMSE':r['val_RMSE'],
                    'selected_test_RMSE':test_rmse,
                    'unstable_divergent':r['divergent']})
result5=pd.DataFrame(result5)
display(result5.round(5))

fig,axs=plt.subplots(1,2,figsize=(13,4))
for r in chosen5:
    axs[0].plot(np.arange(1,len(r['losses'])+1),r['losses'],label=f"{r['method']} lr={r['lr']}")
    axs[1].plot(r['elapsed'],r['losses'],label=f"{r['method']} lr={r['lr']}")
for ax in axs:
    ax.set_ylabel('Full-training objective (MSE/2)');ax.legend();ax.grid(alpha=.3)
axs[0].set(xlabel='Epoch',title='Convergence vs epoch')
axs[1].set(xlabel='Elapsed seconds',title='Convergence vs time')
plt.tight_layout();plt.show()

ls_pred=predict_ols(Xa_te,multi_coef)
ls_rmse=np.sqrt(mean_squared_error(yte,ls_pred))
print('Least-squares test RMSE:',round(ls_rmse,4))
for r in chosen5:
    print(r['method'], 'selected lr',r['lr'], 'test RMSE',
          round(np.sqrt(np.mean((G_te@r['w']+r['b']-yte)**2)),4),
          'updates',r['updates'])
print('One epoch means processing the dataset once; SGD performs one update per example, batch only one.')

## 6. K-Nearest Neighbours — Iris

Implement Euclidean / Manhattan distances and majority vote from scratch. **Deterministic tie rule:** alphabetically smallest class label among tied classes. Evaluate $k=1,3,5,7,9,11$ with and without training-fitted scaling. Choose the configuration on validation accuracy; use test set once for the selected configuration.

In [ ]:
I_tr0=iris_tr.to_numpy(dtype=float);I_va0=iris_va.to_numpy(dtype=float);I_te0=iris_te.to_numpy(dtype=float)
I_scaler=StandardScaler().fit(I_tr0)
I_scaled=[I_scaler.transform(v) for v in [I_tr0,I_va0,I_te0]]
yI_tr,yI_va,yI_te=(v.astype(str).to_numpy() for v in [iy_tr,iy_va,iy_te])

def knn_one(X_train,y_train,x,k,metric):
    diff=np.abs(X_train-x)
    dist=(np.sqrt((diff**2).sum(axis=1)) if metric=='euclidean' else diff.sum(axis=1))
    # Stable tie-break for identical distances: original training-row index.
    inds=np.lexsort((np.arange(len(dist)),dist))[:k]
    labels=y_train[inds]
    vals,counts=np.unique(labels,return_counts=True)  # np.unique sorts labels
    prediction=vals[np.argmax(counts)]  # alphabetical label when vote tied
    return prediction,inds,dist[inds]

def knn_predict(X_train,y_train,X_query,k,metric):
    return np.array([knn_one(X_train,y_train,x,k,metric)[0] for x in X_query])

knn_rows=[]
for scale in [False,True]:
    Xt,Xv = (I_scaled[0],I_scaled[1]) if scale else (I_tr0,I_va0)
    for metric in ['euclidean','manhattan']:
        for k in [1,3,5,7,9,11]:
            pred=knn_predict(Xt,yI_tr,Xv,k,metric)
            knn_rows.append({'scaled':scale,'metric':metric,'k':k,
                             'validation_accuracy':accuracy_score(yI_va,pred)})
knn_table=pd.DataFrame(knn_rows)
display(knn_table.sort_values('validation_accuracy',ascending=False).round(4))
plt.figure(figsize=(10,5))
for (scale,metric),grp in knn_table.groupby(['scaled','metric']):
    plt.plot(grp.k,grp.validation_accuracy,'o-',label=f'{metric}, scaled={scale}')
plt.xlabel('k');plt.ylabel('Validation accuracy');plt.title('KNN accuracy vs k');plt.grid(alpha=.3);plt.legend();plt.show()
# For validation ties, prefer small k, Euclidean, no scaling (deterministic).
knn_best=knn_table.sort_values(['validation_accuracy','k','metric','scaled'],
                               ascending=[False,True,True,True]).iloc[0]
kbest=int(knn_best.k);metricbest=str(knn_best.metric);scaledbest=bool(knn_best.scaled)
IT,IE=(I_scaled[0],I_scaled[2]) if scaledbest else (I_tr0,I_te0)
knn_test=knn_predict(IT,yI_tr,IE,kbest,metricbest)
sk_knn=KNeighborsClassifier(n_neighbors=kbest,metric=metricbest,weights='uniform')
sk_knn.fit(IT,yI_tr)
sk_knn_test=sk_knn.predict(IE)
print('Selected:',{'k':kbest,'distance':metricbest,'scaled':scaledbest})
for label,pred in [('Scratch',knn_test),('scikit-learn',sk_knn_test)]:
    print(label,{'accuracy':accuracy_score(yI_te,pred),
                 'macro_precision':precision_score(yI_te,pred,average='macro',zero_division=0),
                 'macro_recall':recall_score(yI_te,pred,average='macro',zero_division=0),
                 'macro_F1':f1_score(yI_te,pred,average='macro',zero_division=0)})
ConfusionMatrixDisplay.from_predictions(yI_te,knn_test,xticks_rotation=40)
plt.title('Selected KNN: test confusion matrix');plt.tight_layout();plt.show()
for i in range(3):
    prediction,inds,dists=knn_one(IT,yI_tr,IE[i],kbest,metricbest)
    print(f'\nTest sample {i}: actual={yI_te[i]}, predicted={prediction}')
    display(pd.DataFrame({'train_row_index':inds,'distance':dists,
                          'neighbour_label':yI_tr[inds]}).round(4))
    print('Misclassified:',prediction!=yI_te[i])
print('Misclassification, if any, occurs when nearby points from other species outvote the true class.')

## 7. Decision Trees — Breast Cancer

Compare Gini and entropy; depth 2, 3, 5, unrestricted; min leaf 1, 5, 10. Select using validation **malignant-class F1**. Calculate selected root split's parent/child impurity on training samples; visualize 3 levels and extract three root-to-leaf rules.

In [ ]:
# Trees do not require standardization. Median imputation is fit using training data only.
T_tr,T_va,T_te=c_tr0,c_va0,c_te0
features=list(X_cancer.columns)
tree_rows=[]; tree_models={}
for criterion in ['gini','entropy']:
    for depth in [2,3,5,None]:
        for min_leaf in [1,5,10]:
            clf=DecisionTreeClassifier(criterion=criterion,max_depth=depth,
                                       min_samples_leaf=min_leaf,random_state=SEED)
            clf.fit(T_tr,Y_tr)
            key=(criterion,depth,min_leaf);tree_models[key]=clf
            tree_rows.append({'criterion':criterion,'max_depth':depth,'min_samples_leaf':min_leaf,
                'train_accuracy':accuracy_score(Y_tr,clf.predict(T_tr)),
                'train_F1':f1_score(Y_tr,clf.predict(T_tr)),
                'val_accuracy':accuracy_score(Y_va,clf.predict(T_va)),
                'val_F1':f1_score(Y_va,clf.predict(T_va))})
tree_table=pd.DataFrame(tree_rows)
display(tree_table.sort_values('val_F1',ascending=False).round(4))
# Ties: prefer smaller tree depth, larger leaf size, then Gini.
selected_row=sorted(tree_rows,key=lambda r:(-r['val_F1'],
                        (10**9 if r['max_depth'] is None else r['max_depth']),
                        -r['min_samples_leaf'],r['criterion']))[0]
key=(selected_row['criterion'],selected_row['max_depth'],selected_row['min_samples_leaf'])
selected_tree=tree_models[key]
print('Selected by validation F1:',key)
print('Train F1:',round(selected_row['train_F1'],4),'Validation F1:',round(selected_row['val_F1'],4))
print('Large train vs validation gap suggests overfitting; poor scores on both suggest underfitting.')
yp=selected_tree.predict(T_te)
print('Test metrics:',{'accuracy':accuracy_score(Y_te,yp),
    'precision':precision_score(Y_te,yp,zero_division=0),
    'recall':recall_score(Y_te,yp,zero_division=0),
    'F1':f1_score(Y_te,yp,zero_division=0)})
ConfusionMatrixDisplay.from_predictions(Y_te,yp,display_labels=['Benign','Malignant'])
plt.title('Decision tree test confusion matrix');plt.show()

plt.figure(figsize=(19,8))
plot_tree(selected_tree,feature_names=features,class_names=['Benign','Malignant'],
          filled=True,max_depth=2,rounded=True,fontsize=8)
plt.title('Selected decision tree (first 3 levels: depth 0, 1, 2)');plt.show()

# Manually compute selected root impurity from the training observations.
root_feature=selected_tree.tree_.feature[0]
root_threshold=selected_tree.tree_.threshold[0]
assert root_feature>=0, 'Tree has no root split.'
left_mask=T_tr[:,root_feature]<=root_threshold
left_y,right_y=Y_tr[left_mask],Y_tr[~left_mask]

def impurity_manual(y,criterion):
    if len(y)==0: return 0.
    probs=np.bincount(y,minlength=2)/len(y)
    if criterion=='gini': return 1-np.sum(probs**2)
    return -np.sum(probs[probs>0]*np.log2(probs[probs>0]))

parent_imp=impurity_manual(Y_tr,selected_tree.criterion)
left_imp=impurity_manual(left_y,selected_tree.criterion)
right_imp=impurity_manual(right_y,selected_tree.criterion)
weighted=(len(left_y)*left_imp+len(right_y)*right_imp)/len(Y_tr)
reduction=parent_imp-weighted
print('Root:',features[root_feature], '<=',round(root_threshold,5))
print('Parent n:',len(Y_tr),'left n:',len(left_y),'right n:',len(right_y))
print('Parent impurity:',parent_imp,'left:',left_imp,'right:',right_imp)
print('Weighted child impurity:',weighted,'impurity reduction:',reduction)
print('Sklearn tree root impurity:',selected_tree.tree_.impurity[0])
print('Computed parent == sklearn parent:',np.isclose(parent_imp,selected_tree.tree_.impurity[0]))

# Extract three actual root-to-leaf decision rules from fitted tree.
def extract_leaf_rules(clf, names, limit=3):
    t=clf.tree_; results=[]
    def visit(node,clauses):
        if len(results)>=limit: return
        if t.feature[node]==_tree.TREE_UNDEFINED:
            predicted=int(clf.classes_[np.argmax(t.value[node][0])])
            results.append((' AND '.join(clauses) if clauses else 'always',
                            'Malignant' if predicted==1 else 'Benign',
                            t.n_node_samples[node]))
            return
        name=names[t.feature[node]];cut=t.threshold[node]
        visit(t.children_left[node],clauses+[f'{name} <= {cut:.4f}'])
        visit(t.children_right[node],clauses+[f'{name} > {cut:.4f}'])
    visit(0,[])
    return results
for idx,(conditions,pred,n_samples) in enumerate(extract_leaf_rules(selected_tree,features,3),1):
    print(f'Rule {idx}: IF {conditions} THEN {pred} (training samples at leaf: {n_samples})')

## Final notes for submission

- Execute **Run All** in Jupyter/Google Colab and check every table and figure before submission.
- The UCI repository is the source of all datasets (IDs 53, 9 and 17).
- Sampling and split random seed: **42**. Runtime is machine-dependent.
- Figures and test metrics are generated by running the notebook; no numerical results are invented in advance.
- Add your own brief observations if your instructor expects personal interpretation.